<center>
    <font size="5"> Sieci neuronowe i uczenie głębokie<br/>
        <small><em>Studia stacjonarne II stopnia 2025/2026</em><br/>Kierunek: Matematyka stosowana<br>Specjalność: Analityka danych</small>
    </font>
</center>
<br>



# Konwolucyjne sieci neuronowe: Zadanie

## Import bibliotek

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

print("Numpy version:", np.__version__)
print("Tensorflow version:", tf.__version__)

2026-03-18 10:05:26.294244: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-18 10:05:26.334359: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-18 10:05:27.117169: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Numpy version: 2.4.2
Tensorflow version: 2.20.0


## Zadanie

Stwórz i wytrenuj głęboką konwolucyjną sieć neuronową zdolną poprawnie sklasyfikować zdjęcia pochodzące z fragmentu bazy CIFAR-100 (25 losowych klas).

W trakcie realizacji zadania należy wykonań następujące podzadania:
1. Wygenerować dane uczące i testowe (gotowy kod).
2. Zdefiniować model głebokiej konwolucyjnej sieci neuronowej według własnego pomysłu.
3. Wytrenować zdefiniowaną sieć na danych uczących z wyodrębnieniem zbioru walidacyjnego. W procesie uczenia wykorzystać powielanie danych uczących (data augmentation, https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/image/ImageDataGenerator)
4. Ocenić skuteczność modelu na danych testowych.
5. Powtarzać etapy 2-4 aż do osiągnięcia zadowalających wyników.
6. Przeanalizować działanie stworzonej sieci, zestawienie jakie błedy w jakich klasach, zwizualizować wytrenowane filtry i mapy cech.
7. Zastosować occlusions

### Baza CIFAR-100
- Każdemu przydzielony zostaje fragment bazy CIFAR-100 zawierający dane uczące i testowe obrazów pochodzących z 25 klas.
-Klasy wybierane są losowane w zależności od `nr_albumu`.

In [2]:
# podaj swoj nr albumu
nr_albumu = 143200

np.random.seed(nr_albumu)
(train_images_all, train_labels_all), (test_images_all, test_labels_all) = (
    tf.keras.datasets.cifar100.load_data()
)
my_classes = np.random.choice(range(100), size=25, replace=False)
np.random.seed()

mask = train_labels_all[:, 0] == my_classes[0]
mask2 = test_labels_all[:, 0] == my_classes[0]
train_images = train_images_all[mask]
train_labels = np.ones(shape=(mask.sum(), 1)) * 0
test_images = test_images_all[mask2]
test_labels = np.ones(shape=(mask2.sum(), 1)) * 0
for i, cl in enumerate(my_classes[1:]):
    mask = train_labels_all[:, 0] == cl
    train_images = np.concatenate((train_images, train_images_all[mask]))
    train_labels = np.concatenate(
        (train_labels, np.ones(shape=(mask.sum(), 1)) * (i + 1))
    )
    mask2 = test_labels_all[:, 0] == cl
    test_images = np.concatenate((test_images, test_images_all[mask2]))
    test_labels = np.concatenate(
        (test_labels, np.ones(shape=(mask2.sum(), 1)) * (i + 1))
    )

print("Wylosowane klasy:")
print(my_classes)
print("Odpowiadające im nowe etykiety:")
print(np.array(range(25)))

# mieszanie danych
indx = np.arange(0, train_images.shape[0], 1, dtype=np.int32)
np.random.shuffle(indx)
train_images = train_images[indx]
train_labels = train_labels[indx]
indx = np.arange(0, test_images.shape[0], 1, dtype=np.int32)
np.random.shuffle(indx)
test_images = test_images[indx]
test_labels = test_labels[indx]

# normalizacja
train_images = train_images / 255.0
train_labels = np.int64(train_labels)
test_images = test_images / 255.0
test_labels = np.int64(test_labels)
print("Wymiary danych:")
print(train_images.shape, train_labels.shape, test_images.shape, test_labels.shape)

/home/qertal/miniconda3/envs/venv/lib/python3.12/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Wylosowane klasy:
[19 87 64 15 21 88  3 65 78 70 74 14 67 20 50 41 80 66 16 73 11  5 44 77
 63]
Odpowiadające im nowe etykiety:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24]
Wymiary danych:
(12500, 32, 32, 3) (12500, 1) (2500, 32, 32, 3) (2500, 1)


In [3]:
from keras.layers import Conv2D, MaxPool2D, Input, Flatten, Dense
from keras.models import Model

In [4]:
def conv_model():
    x = Input(shape=(32, 32, 3))
    conv = Conv2D(128, kernel_size=3, activation="relu")(
        x
    )  # dodaj warstwę konwolucyjną
    pool = MaxPool2D(pool_size=2)(conv)
    conv = Conv2D(256, kernel_size=3, activation="relu")(pool)
    pool = MaxPool2D(pool_size=2)(conv)
    flat = Flatten()(pool)
    h = Dense(512, activation="relu")(flat)
    # h = Dense(128, activation='relu')(h)
    y = Dense(25, activation="softmax")(h)
    return Model(inputs=x, outputs=y)


model = conv_model()
model.summary()

I0000 00:00:1773824729.609207   48556 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9264 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070, pci bus id: 0000:02:00.0, compute capability: 8.9


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 30, 30, 128)    │         3,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 15, 15, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 13, 13, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     4,719,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 25)             │        12,825 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,030,681 (19.19 MB)

 Trainable params: 5,030,681 (19.19 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
model.compile(
    loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"]
)

In [6]:
from keras.callbacks import ModelCheckpoint

cb = [ModelCheckpoint("model_cibar.keras", monitor="val_accuracy", save_best_only=True)]

In [7]:
BATCH_SIZE = 64
EPOCHS = 20

In [8]:
model.fit(
    train_images,
    train_labels,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2,
    callbacks=cb,
)

Epoch 1/20


2026-03-18 10:05:32.077922: I external/local_xla/xla/service/service.cc:163] XLA service 0x7982ec01cc00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-18 10:05:32.077976: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4070, Compute Capability 8.9
2026-03-18 10:05:32.092312: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-18 10:05:32.187128: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900
2026-03-18 10:05:32.206957: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-18 10:05:32.207089: I e

 35/157 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0778 - loss: 3.1801

I0000 00:00:1773824734.459814   48644 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


150/157 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.1419 - loss: 2.9762

2026-03-18 10:05:35.586052: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_282', 44 bytes spill stores, 44 bytes spill loads



157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.1449 - loss: 2.9663

2026-03-18 10:05:37.642269: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_98', 72 bytes spill stores, 76 bytes spill loads



157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.2095 - loss: 2.7490 - val_accuracy: 0.2648 - val_loss: 2.5234
Epoch 2/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3492 - loss: 2.2263 - val_accuracy: 0.3576 - val_loss: 2.2010
Epoch 3/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.4309 - loss: 1.9367 - val_accuracy: 0.3960 - val_loss: 2.0635
Epoch 4/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.4997 - loss: 1.6859 - val_accuracy: 0.4384 - val_loss: 1.9520
Epoch 5/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5709 - loss: 1.4264 - val_accuracy: 0.4408 - val_loss: 1.9765
Epoch 6/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6411 - loss: 1.1888 - val_accuracy: 0.4420 - val_loss: 2.0049
Epoch 7/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7028 - loss: 0.9695 - val_accuracy: 0.4588 - val_loss: 2.0248
Epoch 8/20
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7786 - loss: 0.7262 - val_accuracy: 0.4460 - val

___

In [9]:
from tensorflow import keras
from keras.utils import image_dataset_from_directory

In [10]:
train_ds = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
train_ds = train_ds.shuffle(len(train_images)).batch(BATCH_SIZE)

In [11]:
from sklearn.model_selection import train_test_split

test_images, val_images, test_labels, val_labels = train_test_split(
    test_images, test_labels, random_state=42, stratify=test_labels
)

In [12]:
val_ds = tf.data.Dataset.from_tensor_slices((val_images, val_labels)).batch(BATCH_SIZE)

In [13]:
def conv_model():
    x = Input(shape=(32, 32, 3))
    conv = Conv2D(128, kernel_size=3, activation="relu")(
        x
    )  # dodaj warstwę konwolucyjną
    pool = MaxPool2D(pool_size=2)(conv)
    conv = Conv2D(256, kernel_size=3, activation="relu")(pool)
    pool = MaxPool2D(pool_size=2)(conv)
    flat = Flatten()(pool)
    h = Dense(512, activation="relu")(flat)
    # h = Dense(128, activation='relu')(h)
    y = Dense(25, activation="softmax")(h)
    return Model(inputs=x, outputs=y)


model = conv_model()
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 30, 30, 128)    │         3,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 15, 15, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 13, 13, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │     4,719,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 25)             │        12,825 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,030,681 (19.19 MB)

 Trainable params: 5,030,681 (19.19 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
import tensorflow as tf
from tensorflow import keras

BATCH_SIZE = 32

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(len(X_train)).batch(BATCH_SIZE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE)

data_augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        keras.layers.RandomZoom(0.1),
        keras.layers.RandomContrast(0.1),
    ]
)

model = keras.Sequential(
    [
        keras.layers.Input(shape=X_train.shape[1:]),
        data_augmentation,
        keras.layers.Rescaling(1.0 / 255),
        keras.layers.Conv2D(32, 3, activation="relu"),
        keras.layers.MaxPooling2D(),
        keras.layers.Conv2D(64, 3, activation="relu"),
        keras.layers.MaxPooling2D(),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(len(set(y_train)), activation="softmax"),
    ]
)

model.compile(
    optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)

model.fit(train_ds, validation_data=val_ds, epochs=10)

NameError: name 'X_train' is not defined